In [ ]:
import traci
import config
from utils import epsilon_by_step, run_episode
from ReplayBuffer import ReplayBuffer
from torch import optim
from config import *
from DQN import DQN
from Action import Action
from sumo_utils import start_sumo
from logger_utils import EpisodeLogger, default_episode_log_path
import sys

start_sumo()

state_dim = 8
action_dim = len(Action)

policy_net = DQN(state_dim, action_dim).to(DEVICE)
target_net = DQN(state_dim, action_dim).to(DEVICE)
target_net.load_state_dict(policy_net.state_dict())
target_net.eval()

optimizer = optim.Adam(policy_net.parameters(), lr=LR)
replay_buffer = ReplayBuffer(BUFFER_CAPACITY)

global_step = 0

# Log file for this run
episode_logger = EpisodeLogger(default_episode_log_path())
print(f"Logging episodes to: {episode_logger.path}")

for episode in range(NUM_EPISODES):

    ep_reward, ep_steps, global_step, end_reason = run_episode(
        policy_net, target_net, optimizer, replay_buffer, global_step
    )


    epsilon = epsilon_by_step(global_step)

    # print(
    #     f"Episode {episode:4d} | "
    #     f"reward={ep_reward:8.2f} | "
    #     f"steps={ep_steps:4d} | "
    #     f"end={end_reason} | "
    #     f"TOTAL_EGO_CRASHES={config.TOTAL_EGO_CRASHES} | "
    #     f"TOTAL_COLLISION_EVENTS={config.TOTAL_COLLISION_EVENTS} | "
    #     f"TOTAL_EGO_COLLISIONS={config.TOTAL_EGO_COLLISIONS} | "
    #     f"TOTAL_EGO_TELEPORTS={config.TOTAL_EGO_TELEPORTS} | "
    #     f"TOTAL_EGO_EMERGENCY_STOPS={config.TOTAL_EGO_EMERGENCY_STOPS}"
    # )

    episode_logger.log(
        episode=episode,
        reward=ep_reward,
        steps=ep_steps,
        end_reason=end_reason,
        total_ego_crashes=config.TOTAL_EGO_CRASHES,
        total_collision_events=config.TOTAL_COLLISION_EVENTS,
        total_ego_collisions=config.TOTAL_EGO_COLLISIONS,
        total_ego_teleports=config.TOTAL_EGO_TELEPORTS,
        total_ego_emergency_stops=config.TOTAL_EGO_EMERGENCY_STOPS,
    )

    # optional checkpoint
    if (episode + 1) % 50 == 0:
        torch.save(policy_net.state_dict(), f"dqn_training/dqn_ego_episode_{episode+1}.pth")

traci.close()

In [ ]:

import sys
import traci

from config import *
from sumo_utils import start_sumo, spawn_ego
# f = open("sim_logs/run.log", "a", buffering=1)
# sys.stdout = f

for r in VALIDATION_ROUTES:
    # traci.start(["sumo-gui", "-c", SUMO_CONFIG])

    start_sumo()
    print(spawn_ego(r))
    for step in range(300):
        try:
            traci.simulationStep()
            if "ego" not in traci.vehicle.getIDList():

                # print(r, "ended at step", step)
                if (step == 0):
                    print(f"{r} Ego vehicle was not added to the simulation.")

                break
        except Exception as e:
            print(f"Error during simulation step {step} for route {r}: {e}")
            break
    traci.close()

In [1]:
from utils import run_loaded_model_on_route
from config import *
# run_loaded_model_on_route("./dqn_training_2/dqn_ego_episode_1000.pth", EGO_ROUTE_POOL[7], True)
run_loaded_model_on_route("./dqn_training_4/dqn_ego_episode_500.pth", EGO_ROUTE_POOL[7], True)

 Retrying in 1 seconds
 Retrying in 1 seconds


(498.00190056683454, 1120, 'arrived')

In [3]:
from utils import validate_model_on_routes
from config import *

scales = [2, 3, 5]

out_path = validate_model_on_routes(
    model_path="dqn_training_4/dqn_ego_episode_500.pth",
    sumo_config=SUMO_CONFIG,
    traffic_scales=scales,
    use_gui=False,
    max_steps=1500,
    out_tsv_path='./dqn_training_4/validation_results.tsv',
)

print(out_path)

 Retrying in 1 seconds
 Retrying in 1 seconds


[2026-04-29T21:17:23.997307+01:00] Warning: Vehicle 'Pertini_20_16.1'; junction collision with vehicle 'Prati_Capraia_30_6', lane=':b4_9_1', gap=-1.00, time=94.00, stage=move.
[2026-04-29T21:17:24.616635+01:00] Warning: Vehicle 'Turati_10_1'; junction collision with vehicle 'bus_140_120', lane=':a27_10_0', gap=-1.00, time=184.00, stage=move.
[2026-04-29T21:17:24.677840+01:00] Warning: Vehicle 'bus_140_120.1'; junction collision with vehicle 'Turati_10_1', lane=':a27_1_0', gap=-1.00, time=191.00, stage=move.
[2026-04-29T21:17:24.747787+01:00] Warning: Vehicle 'Pertini_20_18'; junction collision with vehicle 'Prati_Capraia_50_15', lane=':b4_9_1', gap=-1.00, time=199.00, stage=move.
[2026-04-29T21:17:24.766623+01:00] Warning: Vehicle 'Pertini_20_22.1'; junction collision with vehicle 'Prati_Capraia_50_15', lane=':b4_9_1', gap=-1.00, time=201.00, stage=move.


 Retrying in 1 seconds


[2026-04-29T21:17:28.752159+01:00] Warning: Vehicle 'Pertini_20_16'; junction collision with vehicle 'Prati_Capraia_30_6.1', lane=':b4_9_1', gap=-1.00, time=114.00, stage=move.
[2026-04-29T21:17:29.245466+01:00] Warning: Vehicle 'Turati_10_1'; junction collision with vehicle 'bus_140_120', lane=':a27_10_0', gap=-1.00, time=184.00, stage=move.
[2026-04-29T21:17:29.339819+01:00] Warning: Vehicle 'Turati_10_1'; junction collision with vehicle 'bus_140_120.1', lane=':a27_10_0', gap=-1.00, time=194.00, stage=move.


 Retrying in 1 seconds


...

[2026-04-29T21:29:35.006898+01:00] Warning: Vehicle 'bus_12_0.1'; junction collision with vehicle 'Pertini_20_7.1', lane=':b4_3_0', gap=-1.00, time=71.00, stage=move.
[2026-04-29T21:29:35.033193+01:00] Warning: Vehicle 'Pertini_20_7.3'; junction collision with vehicle 'bus_12_0.1', lane=':b4_9_1', gap=-1.00, time=76.00, stage=move.
[2026-04-29T21:29:35.098965+01:00] Warning: Vehicle 'Pertini_20_9.3'; junction collision with vehicle 'bus_12_0.1', lane=':b4_9_1', gap=-1.00, time=88.00, stage=move.
[2026-04-29T21:29:35.204547+01:00] Warning: Vehicle 'Pertini_20_10.4'; junction collision with vehicle 'bus_12_0.1', lane=':b4_9_1', gap=-1.00, time=104.00, stage=move.
[2026-04-29T21:29:35.221442+01:00] Warning: Vehicle 'Pertini_20_11.1'; junction collision with vehicle 'bus_12_0.1', lane=':b4_9_1', gap=-1.00, time=106.00, stage=move.
[2026-04-29T21:29:35.244283+01:00] Warning: Vehicle 'Pertini_20_11.4'; junction collision with vehicle 'bus_12_0.1', lane=':b4_9_1', gap=-1.00, time=109.00, stag

 Retrying in 1 seconds


[2026-04-29T21:29:40.642885+01:00] Warning: Vehicle 'bus_12_0.1'; junction collision with vehicle 'Pertini_20_7.1', lane=':b4_3_0', gap=-1.00, time=71.00, stage=move.
[2026-04-29T21:29:40.672972+01:00] Warning: Vehicle 'Pertini_20_7.3'; junction collision with vehicle 'bus_12_0.1', lane=':b4_9_1', gap=-1.00, time=76.00, stage=move.
[2026-04-29T21:29:40.742989+01:00] Warning: Vehicle 'Pertini_20_9.3'; junction collision with vehicle 'bus_12_0.1', lane=':b4_9_1', gap=-1.00, time=88.00, stage=move.
[2026-04-29T21:29:40.790368+01:00] Warning: Vehicle 'Pertini_20_10'; junction collision with vehicle 'bus_12_0.1', lane=':b4_9_1', gap=-1.00, time=94.00, stage=move.
[2026-04-29T21:29:40.813085+01:00] Warning: Vehicle 'bus_12_0.1'; junction collision with vehicle 'Pertini_20_9.4', lane=':b4_3_0', gap=-1.00, time=97.00, stage=move.
[2026-04-29T21:29:40.876359+01:00] Warning: Vehicle 'bus_12_0.1'; junction collision with vehicle 'Pertini_20_9.2', lane=':b4_3_0', gap=-1.00, time=104.00, stage=move

./dqn_training_4/validation_results.tsv


[2026-04-29T21:29:43.593867+01:00] Warning: Vehicle 'bus_12_0.3'; junction collision with vehicle 'Pertini_20_19.3', lane=':b4_3_0', gap=-1.00, time=312.00, stage=move.
